In [104]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2

In [105]:
data = pd.read_csv(r'data/data_preprocessed.csv')
print(data.shape)

(274268, 22)


In [106]:
data_dev, data_test = train_test_split(data, test_size=5000/278725, random_state=42)
data_train, data_val = train_test_split(data_dev, test_size=0.2, random_state=42)

print(data_train.shape)
print(data_val.shape)
print(data_test.shape)

(215477, 22)
(53870, 22)
(4921, 22)


In [107]:
X_train = data_train.drop(columns=['precio_pesos_constantes'])
y_train = data_train['precio_pesos_constantes']
X_val = data_val.drop(columns=['precio_pesos_constantes'])
y_val = data_val['precio_pesos_constantes']
X_test = data_test.drop(columns=['precio_pesos_constantes'])
y_test = data_test['precio_pesos_constantes']

print(X_train.shape)
print(y_train.shape)
print(X_val.shape)
print(y_val.shape)
print(X_test.shape)
print(y_test.shape)

(215477, 21)
(215477,)
(53870, 21)
(53870,)
(4921, 21)
(4921,)


In [108]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [109]:
model = Sequential()

# Capa de entrada
model.add(Input(shape=(X_train.shape[1],)))
model.add(Dense(128, activation='relu'))
model.add(BatchNormalization())
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(32, activation='relu'))
model.add(BatchNormalization())
model.add(Dense(1))

In [110]:
model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')

In [111]:
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history = model.fit(X_train_scaled, y_train, epochs=100, batch_size=32, validation_data=(X_val_scaled, y_val), verbose=1, callbacks=[early_stopping])

Epoch 1/100
6734/6734 ━━━━━━━━━━━━━━━━━━━━ 11s 1ms/step - loss: 2187421184.0000 - val_loss: 1901791360.0000
Epoch 2/100
6734/6734 ━━━━━━━━━━━━━━━━━━━━ 9s 1ms/step - loss: 1896308608.0000 - val_loss: 1372208384.0000
Epoch 3/100
6734/6734 ━━━━━━━━━━━━━━━━━━━━ 9s 1ms/step - loss: 1383185536.0000 - val_loss: 866211072.0000
Epoch 4/100
6734/6734 ━━━━━━━━━━━━━━━━━━━━ 9s 1ms/step - loss: 935708864.0000 - val_loss: 580122560.0000
Epoch 5/100
6734/6734 ━━━━━━━━━━━━━━━━━━━━ 9s 1ms/step - loss: 644561664.0000 - val_loss: 351260352.0000
Epoch 6/100
6734/6734 ━━━━━━━━━━━━━━━━━━━━ 9s 1ms/step - loss: 530628000.0000 - val_loss: 388474976.0000
Epoch 7/100
6734/6734 ━━━━━━━━━━━━━━━━━━━━ 9s 1ms/step - loss: 464278592.0000 - val_loss: 314874688.0000
Epoch 8/100
6734/6734 ━━━━━━━━━━━━━━━━━━━━ 9s 1ms/step - loss: 425182080.0000 - val_loss: 271745856.0000
Epoch 9/100
6734/6734 ━━━━━━━━━━━━━━━━━━━━ 9s 1ms/step - loss: 381158016.0000 - val_loss: 274450656.0000
Epoch 10/100
6734/6734 ━━━━━━━━━━━━━━━━━━━━ 10s 1

In [112]:
y_train_pred = model.predict(X_train_scaled)
rmse = root_mean_squared_error(y_train, y_train_pred)
r2 = r2_score(y_train, y_train_pred)

print(f'RMSE: {rmse}')
print(f'R²: {r2}')

6734/6734 ━━━━━━━━━━━━━━━━━━━━ 4s 598us/step
RMSE: 13938.76019283158
R²: 0.8829123134646368


In [113]:
y_val_pred = model.predict(X_val_scaled)
rmse = root_mean_squared_error(y_val, y_val_pred)
r2 = r2_score(y_val, y_val_pred)

print(f'RMSE: {rmse}')
print(f'R²: {r2}')

1684/1684 ━━━━━━━━━━━━━━━━━━━━ 1s 623us/step
RMSE: 14417.100930034954
R²: 0.8703800653612921
